# Tyre Cross-Section Analysis

Compute per-component **area, centroid, volume, surface area, weight, and polar moment of inertia** from 2D DXF tyre cross-sections, and compare design iterations — all inside this notebook (no separate app, no `.py` to run).

**How to use**
1. Run the **Setup** and **Engine** cells once (top to bottom).
2. Then use **Option A** for clean per-layer DXFs, or **Option B** to hand-pick components from a raw drawing.

Runs great in Anaconda / Jupyter. `Cell ▸ Run All` executes the built-in demo end to end.

## 1 · Setup — dependencies (run once)

Safe to re-run; skips anything already installed and **won't crash** if it can't
install. On a locked-down/office PC pip is often blocked — if so, install once
with **conda** from the Anaconda Prompt (the cell prints the exact command):

```
conda install -c conda-forge ezdxf shapely numpy pandas plotly nbformat -y
```
then re-run this cell.

In [ ]:
import sys, subprocess, importlib

REQS = [("ezdxf", "ezdxf>=1.1"), ("shapely", "shapely>=2.0"), ("numpy", "numpy>=1.24"),
        ("pandas", "pandas>=2.0"), ("plotly", "plotly>=5.18"), ("nbformat", "nbformat>=5.0")]

def _missing():
    out = []
    for mod, spec in REQS:
        try:
            importlib.import_module(mod)
        except ImportError:
            out.append(spec)
    return out

_conda_cmd = "conda install -c conda-forge " + " ".join(s.split(">=")[0] for _, s in REQS) + " -y"

missing = _missing()
if not missing:
    print("All dependencies already present.")
else:
    print("Missing:", ", ".join(missing))
    print("Attempting pip install…\n")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-warn-script-location", *missing],
                       capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-2500:])
    if r.returncode != 0 or _missing():
        if r.stderr:
            print("pip error (tail):\n" + r.stderr[-2500:])
        still = _missing()
        if still:
            print("\n" + "=" * 68)
            print("Could not install:", ", ".join(still))
            print("pip is likely blocked on this machine. Install with CONDA instead —")
            print("open the Anaconda Prompt, run this line, then re-run this cell:\n")
            print("   " + _conda_cmd)
            print("=" * 68)
        else:
            print("Installed OK.")
    else:
        print("Installed OK." if not _missing() else "Installed; re-run this cell to confirm.")

## 2 · Engine
All the analysis functions. Run this once; you don't need to read it — the usage cells below are what you edit.

In [ ]:
# ===========================================================================
# Tyre Cross-Section Analysis — engine (notebook edition)
# Same validated math as the app: area/centroid, Pappus volume & surface,
# weight, mass polar MOI of the revolved solid, plus raw-drawing reconstruction.
# ===========================================================================
import math
from dataclasses import dataclass, field
from pathlib import Path

import ezdxf
from ezdxf import path as ezpath
from shapely.geometry import Polygon, MultiPolygon, LineString
from shapely.geometry.polygon import orient
from shapely.ops import polygonize, unary_union
import pandas as pd
import plotly.graph_objects as go

MM3_PER_CM3 = 1000.0
DEFAULT_FLATTEN_TOL = 0.01

# Placeholder specific gravities (g/cm3) — REPLACE with your real compounds.
DEFAULT_SG_TABLE = {
    "TREAD": 1.15, "SIDEWALL": 1.10, "APEX": 1.12, "INNERLINER": 1.20,
    "BELT-1": 1.30, "BELT-2": 1.30, "PLY-1": 1.25, "PLY-2": 1.25,
    "BEAD": 7.85, "BEAD-FILLER": 1.12, "CHAFER": 1.18,
}
FALLBACK_SG = 1.10

COLUMNS = ["file", "component", "area_mm2", "centroid_x", "centroid_y",
           "centroid_distance_mm", "volume_cm3", "weight_g", "surface_area_mm2",
           "polar_MOI", "radius_of_gyration_mm", "sg", "n_holes", "axis_crosses",
           "sg_matched", "pct_of_total_weight"]

_PALETTE = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2",
            "#B279A2", "#EECA3B", "#9D755D", "#BAB0AC", "#FF9DA6"]
_SUPPORTED = {"LWPOLYLINE", "POLYLINE", "SPLINE", "CIRCLE", "ELLIPSE", "ARC", "LINE"}
_ACIS = {"REGION", "3DSOLID", "BODY", "SURFACE"}
_SKIP = {"MTEXT", "TEXT", "DIMENSION", "INSERT", "HATCH", "LEADER"}


def normalise_key(name):
    return name.strip().upper()


def comp_colors(components):
    comps = list(dict.fromkeys(components))
    return {c: _PALETTE[i % len(_PALETTE)] for i, c in enumerate(comps)}


def _rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{alpha})"


def lookup_sg(layer_name, table=None, fallback=FALLBACK_SG):
    src = DEFAULT_SG_TABLE if table is None else table
    lut = {normalise_key(k): float(v) for k, v in src.items()}
    key = normalise_key(layer_name)
    if key in lut:
        return lut[key], True
    if fallback is None:
        raise KeyError(f"No specific gravity for layer {layer_name!r}")
    return fallback, False


# ----- geometry ------------------------------------------------------------
@dataclass
class ComponentGeometry:
    layer: str
    area_mm2: float
    centroid_x: float
    centroid_y: float
    boundary_length_mm: float
    boundary_centroid_x: float
    boundary_centroid_y: float
    r3_moment_mm5: float
    n_holes: int
    polygon: object

    def area_centroid_distance(self, axis_x):
        return abs(self.centroid_x - axis_x)

    def boundary_centroid_distance(self, axis_x):
        return abs(self.boundary_centroid_x - axis_x)


def build_polygon(loops):
    raw = [Polygon(l) for l in loops if len(l) >= 4]
    raw = [p for p in raw if p.is_valid and p.area > 0]
    if not raw:
        raise ValueError("No valid polygon could be built from the supplied loops")
    if len(raw) == 1:
        return orient(raw[0], sign=1.0)
    raw.sort(key=lambda p: p.area, reverse=True)
    shells, holes = [], {}
    for poly in raw:
        placed = False
        for idx, shell in enumerate(shells):
            if shell.contains(poly.representative_point()):
                holes.setdefault(idx, []).append(poly.exterior.coords[:])
                placed = True
                break
        if not placed:
            shells.append(poly)
    parts = [Polygon(shell.exterior.coords[:], holes.get(idx, []))
             for idx, shell in enumerate(shells)]
    poly = parts[0] if len(parts) == 1 else MultiPolygon(parts)
    return orient(poly, sign=1.0) if isinstance(poly, Polygon) else poly


def _iter_rings(polygon):
    polys = [polygon] if isinstance(polygon, Polygon) else list(polygon.geoms)
    for p in polys:
        p = orient(p, sign=1.0)
        yield p.exterior.coords[:], False
        for interior in p.interiors:
            yield interior.coords[:], True


def boundary_centroid(polygon):
    sum_len = sum_x = sum_y = 0.0
    for coords, _ in _iter_rings(polygon):
        for (x0, y0), (x1, y1) in zip(coords[:-1], coords[1:]):
            seg = math.hypot(x1 - x0, y1 - y0)
            if seg == 0:
                continue
            sum_len += seg
            sum_x += seg * (x0 + x1) * 0.5
            sum_y += seg * (y0 + y1) * 0.5
    if sum_len == 0:
        c = polygon.centroid
        return c.x, c.y, 0.0
    return sum_x / sum_len, sum_y / sum_len, sum_len


def r3_area_moment(polygon, axis_x):
    total = 0.0
    for coords, _ in _iter_rings(polygon):
        ring = 0.0
        for (x0, y0), (x1, y1) in zip(coords[:-1], coords[1:]):
            u0, u1 = x0 - axis_x, x1 - axis_x
            dy, du = y1 - y0, u1 - u0
            edge = u0 ** 4 if abs(du) < 1e-14 else (u1 ** 5 - u0 ** 5) / (5.0 * du)
            ring += dy * edge / 4.0
        total += ring
    return abs(total)


def axis_crossing(polygon, axis_x, tol=1e-9):
    xs = [x for coords, _ in _iter_rings(polygon) for x, _ in coords]
    mn, mx = min(xs) - axis_x, max(xs) - axis_x
    return (mn < -tol) and (mx > tol), mn, mx


def geometry_from_polygon(name, polygon, axis_x):
    if isinstance(polygon, Polygon):
        polygon = orient(polygon, sign=1.0)
    c = polygon.centroid
    bcx, bcy, blen = boundary_centroid(polygon)
    n_holes = (len(polygon.interiors) if isinstance(polygon, Polygon)
               else sum(len(p.interiors) for p in polygon.geoms))
    return ComponentGeometry(name, abs(polygon.area), c.x, c.y, blen, bcx, bcy,
                             r3_area_moment(polygon, axis_x), n_holes, polygon)


def compute_geometry(layer, loops, axis_x):
    return geometry_from_polygon(layer, build_polygon(loops), axis_x)


# ----- physics -------------------------------------------------------------
@dataclass
class ComponentResult:
    layer: str
    area_mm2: float
    centroid_x: float
    centroid_y: float
    centroid_distance_mm: float
    volume_cm3: float
    weight_g: float
    surface_area_mm2: float
    polar_moi_g_mm2: float
    radius_of_gyration_mm: float
    sg: float
    sg_matched: bool
    axis_crosses: bool


def compute_physics(geom, axis_x, sg, sg_matched=True):
    d_area = geom.area_centroid_distance(axis_x)
    d_arc = geom.boundary_centroid_distance(axis_x)
    volume_cm3 = (2.0 * math.pi * d_area * geom.area_mm2) / MM3_PER_CM3
    surface = 2.0 * math.pi * d_arc * geom.boundary_length_mm
    weight = volume_cm3 * sg
    rho = sg / MM3_PER_CM3
    polar = 2.0 * math.pi * rho * geom.r3_moment_mm5
    rog = math.sqrt(polar / weight) if weight > 0 else 0.0
    crosses, _, _ = axis_crossing(geom.polygon, axis_x)
    return ComponentResult(geom.layer, geom.area_mm2, geom.centroid_x, geom.centroid_y,
                           d_area, volume_cm3, weight, surface, polar, rog,
                           sg, sg_matched, crosses)


# ----- DXF ingestion (layer path) -----------------------------------------
@dataclass
class ComponentLoop:
    layer: str
    coords: list
    source: str


def _flatten_entity(entity, tol):
    if entity.dxftype() not in _SUPPORTED:
        return None
    try:
        p = ezpath.make_path(entity)
    except (TypeError, ValueError):
        return None
    pts = [(v.x, v.y) for v in p.flattening(distance=tol)]
    return pts if len(pts) >= 2 else None


def _close(pts):
    if len(pts) < 3:
        return pts, False
    a, b = pts[0], pts[-1]
    already = abs(a[0] - b[0]) < 1e-9 and abs(a[1] - b[1]) < 1e-9
    return (pts if already else pts + [pts[0]]), already


def load_dxf(filename, flatten_tol=DEFAULT_FLATTEN_TOL):
    loops, warnings = [], []
    doc = ezdxf.readfile(str(filename))
    for entity in doc.modelspace():
        t = entity.dxftype()
        layer = entity.dxf.layer
        if t in _ACIS:
            warnings.append(f"{layer}: skipped {t} (ACIS blob)")
            continue
        pts = _flatten_entity(entity, flatten_tol)
        if pts is None:
            continue
        pts, _ = _close(pts)
        if len(pts) >= 4:
            loops.append(ComponentLoop(layer, pts, t))
    return loops, warnings


# ----- validation ----------------------------------------------------------
def _is_closed(entity, tol=1e-6):
    t = entity.dxftype()
    if t == "CIRCLE":
        return True
    if t == "ELLIPSE":
        return abs((entity.dxf.end_param - entity.dxf.start_param) - 2 * math.pi) < 1e-3
    if t == "LWPOLYLINE" and entity.closed:
        return True
    if t == "POLYLINE" and entity.is_closed:
        return True
    if t == "SPLINE" and getattr(entity, "closed", False):
        return True
    try:
        pts = list(ezpath.make_path(entity).flattening(0.1))
        if len(pts) >= 3:
            a, b = pts[0], pts[-1]
            return abs(a.x - b.x) < tol and abs(a.y - b.y) < tol
    except Exception:
        pass
    return False


def validate_dxf(filename):
    doc = ezdxf.readfile(str(filename))
    layers = {}
    has_acis = False
    for entity in doc.modelspace():
        t = entity.dxftype()
        layer = entity.dxf.layer
        if t in _ACIS:
            has_acis = True
            continue
        if t not in _SUPPORTED:
            continue
        lr = layers.setdefault(layer, {"counts": {}, "closed": 0, "open": 0})
        lr["counts"][t] = lr["counts"].get(t, 0) + 1
        if _is_closed(entity):
            lr["closed"] += 1
        else:
            lr["open"] += 1
    comp = [n for n in layers if n not in ("0", "Defpoints")]
    all_on_zero = (not comp and any(k in layers for k in ("0", "Defpoints")))
    problems, notes = [], []
    if doc.dxfversion < "AC1015":
        problems.append(f"DXF {doc.dxfversion} older than R2000; arcs may be legacy chains.")
    if has_acis:
        problems.append("Contains REGION/3DSOLID (ACIS) — export raw closed curves instead.")
    if all_on_zero:
        problems.append("All geometry on layer '0' — use the Manual Loop Builder (Option B).")
    for name in comp:
        lr = layers[name]
        if lr["closed"] == 0:
            problems.append(f"Layer '{name}' has no closed loop; run AutoCAD BOUNDARY.")
        elif lr["open"]:
            notes.append(f"Layer '{name}' mixes {lr['closed']} closed + {lr['open']} loose curve(s).")
    return {"filename": Path(filename).name, "dxf_version": doc.dxfversion,
            "layers": layers, "component_layers": comp, "conforms": not problems,
            "problems": problems, "notes": notes, "all_on_layer_zero": all_on_zero}


def print_report(rep):
    print(f"File: {rep['filename']}  ({rep['dxf_version']})")
    print(f"Conforms: {'YES' if rep['conforms'] else 'NO'}")
    for name, lr in sorted(rep["layers"].items()):
        counts = ", ".join(f"{k}:{v}" for k, v in sorted(lr["counts"].items()))
        print(f"  {name!r}: [{counts}]  closed={lr['closed']} open={lr['open']}")
    for p in rep["problems"]:
        print("  PROBLEM:", p)
    for n in rep["notes"]:
        print("  note:", n)


# ----- reconstruction (manual builder path) --------------------------------
@dataclass
class Face:
    id: int
    polygon: object
    area: float
    rep_x: float
    rep_y: float
    cx: float
    cy: float


@dataclass
class Reconstruction:
    filename: str
    curves: list
    faces: list
    xmin: float
    xmax: float
    ymin: float
    ymax: float
    suggested_axis_x: float
    vertical_axis_lines: list = field(default_factory=list)

    def faces_on_side(self, axis_x, side):
        if side == "both":
            return self.faces
        if side == "right":
            return [f for f in self.faces if f.rep_x > axis_x]
        return [f for f in self.faces if f.rep_x < axis_x]


def _flatten_curves(msp, tol):
    curves = []
    for e in msp:
        if e.dxftype() in _SKIP:
            continue
        try:
            pts = [(v.x, v.y) for v in ezpath.make_path(e).flattening(tol)]
        except Exception:
            continue
        if len(pts) >= 2:
            curves.append(pts)
    return curves


def _detect_vertical_axes(msp, tol=0.5):
    xs = []
    for e in msp.query("LINE"):
        s, t = e.dxf.start, e.dxf.end
        if abs(s.x - t.x) <= tol and abs(t.y - s.y) > tol:
            xs.append(round((s.x + t.x) / 2, 3))
    return sorted(set(xs))


def reconstruct(filename, flatten_tol=0.05, min_area=1.0, dedupe=True):
    doc = ezdxf.readfile(str(filename))
    msp = doc.modelspace()
    curves = _flatten_curves(msp, flatten_tol)
    merged = unary_union([LineString(c) for c in curves if len(c) >= 2])
    kept, seen = [], set()
    for poly in polygonize(merged):
        if poly.area < min_area:
            continue
        if dedupe:
            key = (round(poly.area, 2), round(poly.centroid.x, 1), round(poly.centroid.y, 1))
            if key in seen:
                continue
            seen.add(key)
        kept.append(poly)
    kept.sort(key=lambda p: p.area, reverse=True)
    faces = []
    for i, poly in enumerate(kept):
        rp, c = poly.representative_point(), poly.centroid
        faces.append(Face(i, poly, poly.area, rp.x, rp.y, c.x, c.y))
    xs = [x for c in curves for x, _ in c]
    ys = [y for c in curves for _, y in c]
    xmin, xmax = (min(xs), max(xs)) if xs else (0.0, 0.0)
    ymin, ymax = (min(ys), max(ys)) if ys else (0.0, 0.0)
    vaxes = _detect_vertical_axes(msp)
    mid = (xmin + xmax) / 2
    suggested = min(vaxes, key=lambda x: abs(x - mid)) if vaxes else mid
    return Reconstruction(Path(filename).name, curves, faces, xmin, xmax, ymin, ymax,
                          suggested, vaxes)


def assemble_component(faces, face_ids):
    by_id = {f.id: f for f in faces}
    polys = [by_id[i].polygon for i in face_ids if i in by_id]
    if not polys:
        raise ValueError("No faces selected")
    merged = unary_union(polys)
    if merged.geom_type not in ("Polygon", "MultiPolygon"):
        raise ValueError(f"Selected faces did not union into an area ({merged.geom_type})")
    return merged


def faces_table(rec, axis_x):
    rows = []
    for f in rec.faces:
        rows.append({"face_id": f.id, "area_mm2": round(f.area, 1),
                     "cx": round(f.cx, 1), "cy": round(f.cy, 1),
                     "side": "right" if f.rep_x > axis_x else "left"})
    return pd.DataFrame(rows)


# ----- pipeline ------------------------------------------------------------
def _row(file_label, geom, res):
    return {"file": file_label, "component": geom.layer, "area_mm2": res.area_mm2,
            "centroid_x": res.centroid_x, "centroid_y": res.centroid_y,
            "centroid_distance_mm": res.centroid_distance_mm, "volume_cm3": res.volume_cm3,
            "weight_g": res.weight_g, "surface_area_mm2": res.surface_area_mm2,
            "polar_MOI": res.polar_moi_g_mm2, "radius_of_gyration_mm": res.radius_of_gyration_mm,
            "sg": res.sg, "n_holes": geom.n_holes, "axis_crosses": res.axis_crosses,
            "sg_matched": res.sg_matched}


def _finalise(rows):
    df = pd.DataFrame(rows, columns=[c for c in COLUMNS if c != "pct_of_total_weight"])
    total = df["weight_g"].sum()
    df["pct_of_total_weight"] = (df["weight_g"] / total * 100.0) if total else 0.0
    return df[COLUMNS]


def analyse_loops(file_label, loops, axis_x, sg_table=None, fallback_sg=FALLBACK_SG):
    grouped = {}
    for loop in loops:
        grouped.setdefault(loop.layer, []).append(loop.coords)
    rows, geoms, warns = [], [], []
    for layer, layer_loops in grouped.items():
        try:
            geom = compute_geometry(layer, layer_loops, axis_x)
        except ValueError as exc:
            warns.append(f"{layer}: {exc}")
            continue
        sg, matched = lookup_sg(layer, sg_table, fallback_sg)
        if not matched:
            warns.append(f"{layer}: no SG entry; using fallback {sg}")
        res = compute_physics(geom, axis_x, sg, matched)
        if res.axis_crosses:
            warns.append(f"{layer}: crosses rotation axis — Pappus invalid")
        geoms.append(geom)
        rows.append(_row(file_label, geom, res))
    return _finalise(rows), geoms, warns


def analyse_components(file_label, components, axis_x, sg_table=None, fallback_sg=FALLBACK_SG):
    rows, geoms, warns = [], [], []
    for name, poly in components.items():
        geom = geometry_from_polygon(name, poly, axis_x)
        sg, matched = lookup_sg(name, sg_table, fallback_sg)
        if not matched:
            warns.append(f"{name}: no SG entry; using fallback {sg}")
        res = compute_physics(geom, axis_x, sg, matched)
        if res.axis_crosses:
            warns.append(f"{name}: crosses axis — pick faces on ONE side only")
        geoms.append(geom)
        rows.append(_row(file_label, geom, res))
    return _finalise(rows), geoms, warns


def analyse_files(files, axis_x, sg_table=None, fallback_sg=FALLBACK_SG, flatten_tol=0.01):
    frames, geoms_by_file, warnings = [], {}, {}
    for label, path in files.items():
        ax = axis_x[label] if isinstance(axis_x, dict) else axis_x
        loops, w = load_dxf(path, flatten_tol=flatten_tol)
        df, geoms, warns = analyse_loops(label, loops, ax, sg_table, fallback_sg)
        frames.append(df)
        geoms_by_file[label] = geoms
        msgs = list(w) + warns
        if msgs:
            warnings[label] = msgs
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=COLUMNS)
    return combined, geoms_by_file, warnings


# ----- inline plotting -----------------------------------------------------
def _ring_xy(geom):
    polys = [geom.polygon] if isinstance(geom.polygon, Polygon) else list(geom.polygon.geoms)
    for pp in polys:
        x, y = pp.exterior.xy
        yield list(x), list(y)


def overlay_fig(geoms_by_file, baseline, axis_x):
    fig = go.Figure()
    colors = comp_colors([g.layer for gs in geoms_by_file.values() for g in gs])
    for file, geoms in geoms_by_file.items():
        base = file == baseline
        for g in geoms:
            for xs, ys in _ring_xy(g):
                fig.add_trace(go.Scatter(
                    x=xs, y=ys, mode="lines", name=f"{g.layer} ({file})", legendgroup=file,
                    line=dict(color=colors[g.layer], width=2.5 if base else 1.2,
                              dash="solid" if base else "dot"),
                    fill=None if base else "toself",
                    fillcolor=None if base else _rgba(colors[g.layer], 0.18),
                    hovertemplate=f"{g.layer} · {file}<extra></extra>"))
    ax = axis_x if isinstance(axis_x, (int, float)) else list(axis_x.values())[0]
    fig.add_vline(x=ax, line=dict(color="#888", dash="dash"), annotation_text="axis")
    fig.update_layout(title="Cross-section overlay", xaxis_title="x (mm)", yaxis_title="y (mm)",
                      yaxis_scaleanchor="x", height=600, legend=dict(font=dict(size=9)))
    return fig


def moi_fig(df):
    fig = go.Figure()
    for file, g in df.groupby("file", sort=False):
        fig.add_trace(go.Bar(x=g["component"], y=g["polar_MOI"], name=file))
    fig.update_layout(title="Polar MOI by component (r³-driven)", barmode="group",
                      xaxis_title="component", yaxis_title="polar MOI (g·mm²)", height=450)
    return fig


def waterfall_fig(df, baseline, final):
    b = df[df["file"] == baseline].set_index("component")["weight_g"]
    f = df[df["file"] == final].set_index("component")["weight_g"]
    comps = list(dict.fromkeys(list(b.index) + list(f.index)))
    deltas = [f.get(c, 0.0) - b.get(c, 0.0) for c in comps]
    fig = go.Figure(go.Waterfall(
        orientation="v", measure=["relative"] * len(comps) + ["total"],
        x=comps + ["NET Δ"], y=deltas + [0],
        decreasing=dict(marker=dict(color="#2EA043")),
        increasing=dict(marker=dict(color="#E45756")),
        totals=dict(marker=dict(color="#4C78A8")),
        text=[f"{d:+.0f}" for d in deltas] + [f"{sum(deltas):+.0f}"], textposition="outside"))
    fig.update_layout(title=f"Weight waterfall: {baseline} → {final} (g)",
                      yaxis_title="Δ weight (g)", height=460)
    return fig


def face_map_fig(rec, axis_x, side="both", assignments=None):
    """Numbered candidate-face map. Read the numbers, then map them to
    components in an ASSIGNMENTS dict. Assigned faces are coloured."""
    assignments = assignments or {}
    colors = comp_colors(list(assignments) or ["_"])
    face_comp = {fid: comp for comp, ids in assignments.items() for fid in ids}
    faces = rec.faces_on_side(axis_x, side)
    fig = go.Figure()
    for c in rec.curves:
        xs, ys = zip(*c)
        fig.add_trace(go.Scatter(x=list(xs), y=list(ys), mode="lines",
                                 line=dict(color="rgba(120,120,120,0.35)", width=0.6),
                                 hoverinfo="skip", showlegend=False))
    for f in faces:
        comp = face_comp.get(f.id)
        fill = _rgba(colors.get(comp, "#888"), 0.45) if comp else "rgba(180,180,180,0.25)"
        line = colors.get(comp, "#666") if comp else "rgba(150,150,150,0.7)"
        xs, ys = f.polygon.exterior.xy
        fig.add_trace(go.Scatter(x=list(xs), y=list(ys), mode="lines", fill="toself",
                                 fillcolor=fill, line=dict(color=line, width=1),
                                 hoverinfo="skip", showlegend=False))
        fig.add_annotation(x=f.rep_x, y=f.rep_y, text=str(f.id), showarrow=False,
                           font=dict(size=11, color="black"),
                           bgcolor="rgba(255,255,255,0.7)")
    fig.add_vline(x=axis_x, line=dict(color="#0a84ff", dash="dash"), annotation_text="axis")
    fig.update_layout(title=f"Candidate faces — {side} half (numbers = face_id)",
                      xaxis_title="x (mm)", yaxis_title="y (mm)",
                      yaxis_scaleanchor="x", height=700)
    return fig


# ----- sample DXF generators (so the notebook is self-contained) -----------
def _sample_baseline():
    return {
        "TREAD": [(150, 300, 0), (230, 300, 0), (230, 318, -0.15), (150, 318, 0)],
        "BELT-1": [(152, 292, 0), (228, 292, 0), (228, 299, 0), (152, 299, 0)],
        "BELT-2": [(155, 284, 0), (225, 284, 0), (225, 291, 0), (155, 291, 0)],
        "SIDEWALL": [(150, 300, 0), (158, 300, 0.2), (172, 150, 0), (172, 120, 0),
                     (160, 120, 0), (160, 150, -0.2), (146, 300, 0)],
        "PLY-1": [(156, 300, 0), (159, 300, 0.15), (168, 130, 0), (168, 118, 0),
                  (164, 118, 0), (164, 130, -0.15), (153, 300, 0)],
        "APEX": [(160, 118, 0), (170, 118, 0), (163, 175, 0)],
        "BEAD": [(160, 108, 0.414), (170, 108, 0.414), (170, 118, 0.414), (160, 118, 0.414)],
    }


def _sample_iter1():
    b = _sample_baseline()
    b["TREAD"] = [(150, 300, 0), (230, 300, 0), (230, 313, -0.15), (150, 313, 0)]
    b["APEX"] = [(161, 120, 0), (169, 120, 0), (164, 182, 0)]
    return b


def _write_profile(components, path):
    doc = ezdxf.new("R2018")
    doc.header["$INSUNITS"] = 4
    msp = doc.modelspace()
    for layer_name, verts in components.items():
        if layer_name not in doc.layers:
            doc.layers.add(layer_name)
        pts = [(x, y, 0.0, 0.0, bulge) for (x, y, bulge) in verts]
        msp.add_lwpolyline(pts, format="xyseb", close=True, dxfattribs={"layer": layer_name})
    doc.saveas(str(path))


def make_samples(out_dir="sample_data"):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    _write_profile(_sample_baseline(), out / "baseline.dxf")
    _write_profile(_sample_iter1(), out / "iteration1.dxf")
    return out / "baseline.dxf", out / "iteration1.dxf"


def make_raw_sample(path="sample_data/raw_full_section.dxf"):
    """A synthetic NON-conforming full section: everything on layer 0 as open
    LINEs (mirror halves + drawn centreline). Demonstrates Option B."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    doc = ezdxf.new("R2018")
    msp = doc.modelspace()
    axis = 100.0

    def open_rect(x0, y0, x1, y1):
        for a, b in [((x0, y0), (x1, y0)), ((x1, y0), (x1, y1)),
                     ((x1, y1), (x0, y1)), ((x0, y1), (x0, y0))]:
            msp.add_line(a, b)

    # right half: three stacked component bands
    open_rect(120, 40, 150, 55)   # ~ tread band
    open_rect(122, 20, 148, 38)   # ~ belt band
    open_rect(125, 0, 145, 18)    # ~ sidewall/bead band
    # mirrored left half about x=100
    open_rect(50, 40, 80, 55)
    open_rect(52, 20, 78, 38)
    open_rect(55, 0, 75, 18)
    # drawn vertical centreline (rotation axis)
    msp.add_line((100, -8), (100, 63))
    doc.saveas(str(path))
    return Path(path)

In [ ]:
import plotly.io as pio
# Render Plotly charts inline in the notebook.
try:
    pio.renderers.default = "notebook"
except Exception:
    pass
print("Engine loaded.")

---
## Option A — Layer-based files (each component = one closed curve on its own named layer)

Fill in `FILES` (label → path) and the rotation-axis X. The cell below runs the whole analysis and shows the results table and charts.

*Don't have a file handy?* The next cell writes two demo DXFs you can point at.

In [ ]:
# Optional: generate demo DXFs to try Option A immediately.
b, i = make_samples("sample_data")
print("Wrote", b, "and", i)

In [ ]:
# ---- EDIT THESE ----
FILES = {
    "baseline": "sample_data/baseline.dxf",
    "iteration1": "sample_data/iteration1.dxf",
}
BASELINE = "baseline"      # which label is the baseline
AXIS_X   = 0.0             # rotation axis X-coordinate (mm); demo files use 0
SG_TABLE = dict(DEFAULT_SG_TABLE)   # edit values (g/cm3) here, e.g. SG_TABLE["TREAD"] = 1.16
# --------------------

# Validate first
for label, path in FILES.items():
    rep = validate_dxf(path)
    print(f"[{label}] conforms={rep['conforms']}")
    for p in rep["problems"]:
        print("   PROBLEM:", p)

df, geoms_by_file, warnings = analyse_files(FILES, AXIS_X, sg_table=SG_TABLE)
for lbl, msgs in warnings.items():
    for m in msgs:
        print(f"[warn] {lbl}: {m}")

# Per-component table
show = ["file","component","area_mm2","weight_g","centroid_distance_mm",
        "volume_cm3","surface_area_mm2","polar_MOI","pct_of_total_weight"]
display(df[show].round(2))

# Tire totals
print("\nTotal weight per file (g):")
print(df.groupby("file")["weight_g"].sum().round(2))

In [ ]:
# Charts (Option A)
overlay_fig(geoms_by_file, BASELINE, AXIS_X).show()
moi_fig(df).show()
others = [l for l in FILES if l != BASELINE]
if others:
    waterfall_fig(df, BASELINE, others[-1]).show()

---
## Option B — Manual loop builder (raw drawings, e.g. everything on layer 0)

For files that **don't** follow the layer convention. Workflow:

1. Point `RAW_DXF` at your file and set `AXIS_X` / `SIDE`.
2. Run the reconstruction cell — it shows a **numbered map of candidate faces** and a table.
3. In the **assignment cell**, map face numbers to component names, e.g.
   `ASSIGNMENTS = {"SIDEWALL": [3, 4], "BEAD": [6]}`.
4. Run the compute cell.

The big central face is usually the air cavity — don't assign it. Work **one half** (right/left) so Pappus distances stay correct. If a thin band is missing, lower `MIN_AREA`.

*No raw file handy?* The next cell writes a synthetic one (mirror halves on layer 0).

In [ ]:
# Optional: write a synthetic non-conforming raw drawing to demo Option B.
raw = make_raw_sample("sample_data/raw_full_section.dxf")
print("Wrote", raw)
print("Validator says:")
print_report(validate_dxf(raw))

In [ ]:
# ---- EDIT THESE ----
RAW_DXF  = "sample_data/raw_full_section.dxf"
MIN_AREA = 1.0       # drop face slivers below this (mm^2); lower if a band is missing
SIDE     = "right"   # "right", "left" or "both" — analyse ONE symmetric half
# --------------------

rec = reconstruct(RAW_DXF, min_area=MIN_AREA)
AXIS_X = round(rec.suggested_axis_x, 3)     # auto-detected; override if needed
print(f"{len(rec.curves)} curves -> {len(rec.faces)} candidate faces.  Suggested axis X = {AXIS_X}")

# Table of faces to pick from
display(faces_table(rec, AXIS_X))

# Numbered face map — read the numbers, then assign them below.
face_map_fig(rec, AXIS_X, side=SIDE).show()

In [ ]:
# ---- ASSIGN faces to components (EDIT THIS) ----
# Map component name -> list of face_id numbers from the map above.
# Example for the synthetic demo (top-3 right-half bands):
_right = rec.faces_on_side(AXIS_X, SIDE)
ASSIGNMENTS = {
    "TREAD":    [_right[0].id],
    "BELT-1":   [_right[1].id],
    "SIDEWALL": [_right[2].id],
}
# For your real file, replace with explicit ids, e.g.:
#   ASSIGNMENTS = {"TREAD":[0], "BELT-1":[1], "SIDEWALL":[2,3], "BEAD":[5]}

SG_TABLE = dict(DEFAULT_SG_TABLE)   # edit densities (g/cm3) as needed
# ------------------------------------------------

# Re-draw the map with your assignments coloured in, so you can sanity-check.
face_map_fig(rec, AXIS_X, side=SIDE, assignments=ASSIGNMENTS).show()

components = {name: assemble_component(rec.faces, ids) for name, ids in ASSIGNMENTS.items()}
df_m, geoms_m, warns_m = analyse_components("manual", components, AXIS_X, sg_table=SG_TABLE)
for w in warns_m:
    print("[warn]", w)

show = ["component","area_mm2","centroid_distance_mm","volume_cm3",
        "weight_g","surface_area_mm2","polar_MOI","pct_of_total_weight"]
display(df_m[show].round(2))
print("Total weight (one half, g):", round(df_m["weight_g"].sum(), 1))

overlay_fig({"manual": geoms_m}, "manual", AXIS_X).show()
moi_fig(df_m).show()

In [ ]:
# Save results to CSV (opens in Excel)
df_m.to_csv("manual_results.csv", index=False)
print("Wrote manual_results.csv")

---
### Notes
- **Specific gravities are placeholders** — edit `SG_TABLE` with your real compound densities before trusting weights.
- **Units:** drawing mm; area mm², volume cm³, weight g, polar MOI g·mm².
- **Polar MOI** is the mass moment of the revolved solid about the axis (`2π·ρ·∫r³dA`), kept separate from weight because a component can lose weight yet gain MOI if it moves outward.
- The math is validated against a revolved rectangle (= hollow cylinder), whose volume/surface/MOI have known closed forms.